In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("/Users/aaryanbasnet/Desktop/Basic lab/jena_climate_2009_2016.csv")

df_hourly = df[5::6].copy()


data_matrix = df_hourly.iloc[:, 1:].values 
print(f"Original shape: {df.shape}")
print(f"Hourly shape: {data_matrix.shape}")

Original shape: (420551, 15)
Hourly shape: (70091, 14)


In [ ]:
n = len(data_matrix)
train_df = data_matrix[0:int(n*0.7)]
val_df = data_matrix[int(n*0.7):int(n*0.9)]
test_df = data_matrix[int(n*0.9):]

scaler = StandardScaler()
scaler.fit(train_df)

train_data = scaler.transform(train_df)
val_data = scaler.transform(val_df)
test_data = scaler.transform(test_df)

In [ ]:
class JenaWindowDataset(Dataset):
    def __init__(self, data, lookback=120, delay=24):
        self.data = data
        self.lookback = lookback
        self.delay = delay

    def __len__(self):
        return len(self.data) - self.lookback - self.delay

    def __getitem__(self, i):
        x = self.data[i : i + self.lookback]
        y = self.data[i + self.lookback + self.delay - 1, 1] 
        
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


train_dataset = JenaWindowDataset(train_data)
val_dataset = JenaWindowDataset(val_data)

In [ ]:
BATCH_SIZE = 128

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

example_x, example_y = next(iter(train_loader))
print(f"Batch Shape (X): {example_x.shape}")
print(f"Batch Shape (Y): {example_y.shape}")

Batch Shape (X): torch.Size([128, 120, 14])
Batch Shape (Y): torch.Size([128])
